In [1]:
# IMPORTS

import json
import os
import pickle
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

import warnings
warnings.filterwarnings("ignore")


In [2]:
# PATHS

root_path = Path.cwd().resolve()
if root_path.name == "notebooks":
    root_path = root_path.parent

data_dir = root_path / "data"
artifact_dir = root_path / "artifacts"
result_dir = root_path / "results"

for directory in [data_dir, artifact_dir, result_dir]:
    directory.mkdir(parents=True, exist_ok=True)

raw_data_path = data_dir / "transactions.csv"
downsampled_data_path = data_dir / "downsampled_transactions.csv"
NON_FRAUD_TO_FRAUD_RATIO = 5
preprocessing_artifact_path = artifact_dir / "preprocessing_metadata.pkl"
preprocessing_manifest_path = artifact_dir / "preprocessing_manifest.json"


In [3]:
# PREPROCESSING HELPERS

FEATURE_COLS = [
    "log_amount",
    "error_orig",
    "error_dest",
    "hour",
    "is_night",
    "is_high_amount",
    "type_encoded",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest",
]

TYPE_MAPPING = {"TRANSFER": 0, "CASH_OUT": 1}
RAW_NUMERIC_COLS = [
    "step",
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest",
]


def clean_raw_transactions(df):
    df = df.copy()

    for col in RAW_NUMERIC_COLS:
        if col not in df.columns:
            raise ValueError(f"Missing required column: {col}")
        df[col] = pd.to_numeric(df[col], errors="coerce")

    if "type" not in df.columns:
        raise ValueError("Missing required column: type")

    if "isFraud" in df.columns:
        df["isFraud"] = pd.to_numeric(df["isFraud"], errors="coerce").astype("Int64")

    df = df.dropna(subset=["type"] + RAW_NUMERIC_COLS)
    return df


def fit_preprocessing_metadata(df):
    df = clean_raw_transactions(df)
    supported = df[df["type"].isin(TYPE_MAPPING)]

    if supported.empty:
        raise ValueError("No TRANSFER or CASH_OUT rows available for training.")

    return {
        "type_mapping": TYPE_MAPPING.copy(),
        "high_amount_threshold": float(supported["amount"].quantile(0.99)),
    }


def _binary_indicator(series):
    numeric = pd.to_numeric(series, errors="coerce")
    text = series.astype(str).str.strip().str.lower()
    return numeric.fillna(text.isin(["true", "yes", "1"]).astype(int)).astype(int)


def class_distribution_dict(series):
    counts = series.value_counts(dropna=False).sort_index().astype(int).to_dict()
    return {str(k): int(v) for k, v in counts.items()}


def engineer_features(df, high_amount_threshold, type_mapping=None):
    df = clean_raw_transactions(df)
    type_mapping = TYPE_MAPPING if type_mapping is None else type_mapping

    df = df[df["type"].isin(type_mapping)].copy()

    df["log_amount"] = np.log1p(df["amount"].clip(lower=0))
    df["error_orig"] = df["oldbalanceOrg"] - df["amount"] - df["newbalanceOrig"]
    df["error_dest"] = df["oldbalanceDest"] + df["amount"] - df["newbalanceDest"]
    df["hour"] = (df["step"] % 24).astype(int)
    df["is_night"] = ((df["hour"] >= 22) | (df["hour"] <= 5)).astype(int)
    df["is_high_amount"] = (df["amount"] > high_amount_threshold).astype(int)
    df["type_encoded"] = df["type"].map(type_mapping).astype(int)

    if "nameDest" in df.columns:
        df["is_merchant_dest"] = df["nameDest"].astype(str).str.startswith("M").astype(int)
    elif "is_merchant_dest" in df.columns:
        df["is_merchant_dest"] = _binary_indicator(df["is_merchant_dest"])
    else:
        df["is_merchant_dest"] = 0

    for col in FEATURE_COLS + ["is_merchant_dest"]:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

    return df


In [4]:
# LOAD DATA

df = pd.read_csv(data_dir / "transactions.csv")

In [5]:
# BASIC CLEANING

df = clean_raw_transactions(df)

In [6]:
# PREPROCESSING METADATA

preprocessing_metadata = fit_preprocessing_metadata(df)
preprocessing_metadata.update(
    {
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "source_data": str(raw_data_path.relative_to(root_path)),
        "raw_row_count": int(len(df)),
        "raw_class_distribution": class_distribution_dict(df["isFraud"]),
        "feature_columns": FEATURE_COLS,
        "target_column": "isFraud",
        "supported_transaction_types": list(TYPE_MAPPING.keys()),
        "notes": "Raw data remains in data/. Fit preprocessing metadata on training data in model_training.ipynb before model training.",
    }
)

with open(preprocessing_artifact_path, "wb") as f:
    pickle.dump(preprocessing_metadata, f)

with open(preprocessing_manifest_path, "w", encoding="utf-8") as f:
    json.dump(preprocessing_metadata, f, indent=2)

print("Saved preprocessing metadata:", preprocessing_artifact_path)
print("Raw class distribution:", preprocessing_metadata["raw_class_distribution"])


Saved preprocessing metadata: C:\Project\Transactional-Fraud-Detection\artifacts\preprocessing_metadata.pkl
Raw class distribution: {'0': 6354407, '1': 8213}


In [7]:
# FILTER TRANSACTION TYPES

df = df[df["type"].isin(preprocessing_metadata["type_mapping"].keys())].copy()

filtered_row_count = int(len(df))
filtered_class_distribution = class_distribution_dict(df["isFraud"])

fraud_df = df[df["isFraud"] == 1]
non_fraud_df = df[df["isFraud"] == 0]
non_fraud_sample_size = min(len(non_fraud_df), len(fraud_df) * NON_FRAUD_TO_FRAUD_RATIO)
non_fraud_sample = non_fraud_df.sample(n=non_fraud_sample_size, random_state=42)

downsampled_df = pd.concat([fraud_df, non_fraud_sample], ignore_index=True)
downsampled_df = downsampled_df.sample(frac=1, random_state=42).reset_index(drop=True)
downsampled_df.to_csv(downsampled_data_path, index=False)

preprocessing_metadata.update(
    {
        "filtered_supported_row_count": filtered_row_count,
        "filtered_supported_class_distribution": filtered_class_distribution,
        "downsampled_data": str(downsampled_data_path.relative_to(root_path)),
        "downsampled_row_count": int(len(downsampled_df)),
        "downsampled_class_distribution": class_distribution_dict(downsampled_df["isFraud"]),
        "non_fraud_to_fraud_ratio": NON_FRAUD_TO_FRAUD_RATIO,
        "notes": "Raw data remains in data/. Downsampled supported transactions are saved for lightweight training.",
    }
)

with open(preprocessing_artifact_path, "wb") as f:
    pickle.dump(preprocessing_metadata, f)

with open(preprocessing_manifest_path, "w", encoding="utf-8") as f:
    json.dump(preprocessing_metadata, f, indent=2)

df = downsampled_df.copy()

print("Saved downsampled transactions:", downsampled_data_path)
print("Filtered supported class distribution:", filtered_class_distribution)
print("Downsampled class distribution:", preprocessing_metadata["downsampled_class_distribution"])


Saved downsampled transactions: C:\Project\Transactional-Fraud-Detection\data\downsampled_transactions.csv
Filtered supported class distribution: {'0': 2762196, '1': 8213}
Downsampled class distribution: {'0': 41065, '1': 8213}


In [8]:
# FEATURE ENGINEERING

df = engineer_features(
    df,
    preprocessing_metadata["high_amount_threshold"],
    preprocessing_metadata["type_mapping"],
)


In [9]:
# FINAL FEATURE SELECTION

features = FEATURE_COLS

X = df[features]
y = df["isFraud"]

In [10]:
# PREPROCESSING SUMMARY

processed_profile = {
    "processed_row_count": int(len(df)),
    "processed_class_distribution": class_distribution_dict(y),
    "fraud_positive_rate": float(y.mean()),
    "feature_count": int(len(features)),
}

with open(result_dir / "preprocessing_profile.json", "w", encoding="utf-8") as f:
    json.dump(processed_profile, f, indent=2)

print(processed_profile)


{'processed_row_count': 49278, 'processed_class_distribution': {'0': 41065, '1': 8213}, 'fraud_positive_rate': 0.16666666666666666, 'feature_count': 11}
